<a href="https://colab.research.google.com/github/dimitrod/ehu_nlp_dimathina/blob/Huggingface_tutorials/Huggingface_SimpleRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing and importing packages

In [27]:
!pip install langchain langchain-openai langchain-community faiss-cpu tiktoken datasets bitsandbytes transformers sentence_transformers

In [28]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
#from langchain_community.vectorstores import FAISS
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import VectorStoreRetriever
from langchain.chains import RetrievalQA
from datasets import load_dataset
from tqdm import tqdm
import os

Loading and setting up dataset for database

In [29]:
%time validation_set = load_dataset('trivia_qa', name="rc.wikipedia", split="validation")

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

CPU times: user 662 ms, sys: 45.1 ms, total: 707 ms
Wall time: 2.68 s


In [30]:
print(validation_set)

Dataset({
    features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
    num_rows: 7993
})


In [31]:
validation_set = validation_set["entity_pages"]

Creating text splitter to chunk each document into managable pieces


*   chunk_size: size of text block
*   chunk_overlap: Amount of characters which overlap between two consecutive chunks


(Chunk size and overlap could be used as possible hyper parameters for the retriever model)

In [42]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    length_function=len,
)

Splitting all documents into chunks. Currently metadata is included for each chunk. Can be removed to save space

In [46]:
def create_chunks(data_set, amount) :
  '''
  Splits the documents of the dataset into chunks

  Parameters:
  data_set(list): the used dataset
  amount(int): amount of documents to be split

  Returns:
  documents(list): list of chunks of the documents
  '''
  documents = []

  #tqdm ist ein tool um einen Ladebalken anzuzeigen während der Code ausgeführt wird. Nicht unbedingt notwendig aber hilfreich
  for i in tqdm (range(amount), desc="Loading wiki_context") :
    docs = data_set[i]

    temp = ""
    for doc in docs["wiki_context"] :
      temp += doc

    texts = text_splitter.create_documents([temp])
    for text in texts :
      text.metadata["source"] = docs["title"]     #Line can be deleted to remove metadata
      documents.append(text)
  return documents

train_documents = create_chunks(validation_set, len(validation_set))

Loading wiki_context: 100%|██████████| 7993/7993 [02:49<00:00, 47.24it/s]


Load Model for embedding



In [36]:
model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")



---



Create Index **-> Research**

(Hier bin ich mir noch nicht so ganz sicher wie das mit dem indexing funktioniert. Weiß aber auch nicht wie wichtig das für das erstellen der Datenbank ist. Hab mir das von dieser Quelle abgeguckt: https://api.python.langchain.com/en/latest/vectorstores/langchain_community.vectorstores.faiss.FAISS.html)

In [37]:
import faiss

index = faiss.IndexFlatL2(len(model.embed_query("hello world")))


Create database **-> Research**

In [38]:
from langchain_community.docstore.in_memory import InMemoryDocstore

vector_store = FAISS(
    embedding_function=model,
    index=index,
    docstore= InMemoryDocstore(),
    index_to_docstore_id={}
)



---



Load database in one go

In [49]:
def create_database(train_documents, amount, model):
  '''
  Creates database in one go

  train_documents(list): list of chunked documents
  amount(int): amount chunked documents added to database
  model: embedding model

  Returns:
  db: database
  '''
  docs = []
  for i in range(amount):
    docs.append(train_documents[i])
  return FAISS.from_documents(docs, model)

db = create_database(train_documents, 10, model)


Save Database locally

In [50]:
db.save_local("faiss_index")

Test retrieval of text passages with test query

In [51]:
query = "Which Lloyd Webber musical premiered in the US on 10th December 1993?"
answer = db.similarity_search(query)
for a in answer :
  print(a.page_content)

Andrew Lloyd Webber, Baron Lloyd-Webber   (born 22 March 1948) is an English composer and impresario of musical theatre.
Early life

Andrew Lloyd Webber was born in Kensington, London, the elder son of William Lloyd Webber (1914–1982), a composer and organist, and Jean Hermione Johnstone (1921–1993), a violinist and pianist.  His younger brother, Julian Lloyd Webber, is a renowned solo cellist.
His company, the Really Useful Group, is one of the largest theatre operators in London. Producers in several parts of the UK have staged productions, including national tours, of the Lloyd Webber musicals under licence from the Really Useful Group. Lloyd Webber is also the president of the Arts Educational Schools London, a performing arts school located in Chiswick, West London. He is involved in a number of charitable activities, including the Elton John AIDS Foundation, Nordoff Robbins,
Prostate Cancer UK and War Child. In 1992 he set up the Andrew Lloyd Webber Foundation which supports the 

In [ ]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# **LLM Chain starts here**

In [26]:
pip install -U bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "HuggingFaceH4/zephyr-7b-beta"

#bnb_config = BitsAndBytesConfig(
#    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
#)

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00008.safetensors:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00003-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

model-00004-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from transformers import pipeline
from langchain_core.output_parsers import StrOutputParser

text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=True,
    max_new_tokens=400,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<|system|>
Answer the question based on your knowledge. Use the following context to help:

{context}

</s>
<|user|>
{question}
</s>
<|assistant|>

 """

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

llm_chain = prompt | llm | StrOutputParser()

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
<ipython-input-58-3303a33a3a94>:17: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_generation_pipeline)


In [ ]:
from langchain_core.runnables import RunnablePassthrough

retriever = db.as_retriever()

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain

In [ ]:
question = "What age do players in the Junior League Baseball Division usually have?"

In [ ]:
llm_chain.invoke({"context": "", "question": question})

"\n<|system|>\nAnswer the question based on your knowledge. Use the following context to help:\n\n\n\n</s>\n<|user|>\nWhat age do players in the Junior League Baseball Division usually have?\n</s>\n<|assistant|>\n\n  Players in the Junior League Baseball division typically range in age from 13 to 16 years old, depending on the specific league's age cutoff. This division is designed for older youth baseball players who have outgrown the Little League Baseball division but are not yet ready for the more advanced play of high school or travel ball leagues. The focus in this division is on developing fundamental skills, sportsmanship, and teamwork, as well as preparing players for higher levels of competition."

In [ ]:
rag_chain.invoke(question)

'\n<|system|>\nAnswer the question based on your knowledge. Use the following context to help:\n\n[Document(metadata={}, page_content="The Minor League Baseball division is generally for children ages 7–11, with local leagues given the option to allow 6-year-old children to try out. Local leagues are permitted to further divide the Minor League division based on player age and/or experience, and often consist of coach-pitch (i.e., the batter\'s coach lightly pitching the ball) or machine-pitch at lower levels, with defensive players pitching at higher levels.\\n\\n9–10 Year Olds"), Document(metadata={}, page_content=\'"The Junior League Baseball Division is a program for boys and girls ages 13–14, using a conventional 90 ft diamond with a pitching distance of 60 ft. (A modified diamond is available during the regular season.) The local league has an option to choose a Tournament Team (or "All Stars") of 13-14-year-olds from within this division (and/or from within the Senior League Div